# Interface do chatbot acadêmico

Vamos integrar o RAG a uma interface de conversa, exibindo
as respostas e os documentos consultados, com páginas,
links e trechos.

In [2]:
from pathlib import Path

print("Pasta atual:", Path.cwd())
print("Modelo carregado:", "modelo_llm" in globals())
print("Trechos v3 na memória:", "trechos_atualizados" in globals())
print("Embeddings v3 na memória:", "embeddings_atualizados" in globals())

for pasta in [
    Path("/content/projeto-pln/data/processed"),
    Path("/content"),
]:
    print(f"\nArquivos em {pasta}:")
    if pasta.exists():
        for arquivo in pasta.iterdir():
            print(arquivo.name)
    else:
        print("Pasta inexistente")

Pasta atual: /content
Modelo carregado: False
Trechos v3 na memória: False
Embeddings v3 na memória: False

Arquivos em /content/projeto-pln/data/processed:
Pasta inexistente

Arquivos em /content:
.config
sample_data


In [3]:
from google.colab import files

arquivos_interface = files.upload()

Saving base_rag_v3.zip to base_rag_v3.zip


In [4]:
from pathlib import Path
from zipfile import ZipFile
from io import BytesIO
import csv
import hashlib
import numpy as np
import torch

raiz = Path("/content/projeto-pln")
pasta_dados = raiz / "data" / "processed"

arquivos_esperados = {
    "data/processed/trechos_v3.csv",
    "data/processed/embeddings_trechos_v3.npy",
    "data/processed/indice_embeddings_v3.csv",
    "data/processed/configuracao_trechos_v3.csv",
}

with ZipFile(BytesIO(
    arquivos_interface["base_rag_v3.zip"]
)) as pacote:
    assert set(pacote.namelist()) == arquivos_esperados
    assert pacote.testzip() is None
    pacote.extractall(raiz)


def ler_csv(caminho):
    with caminho.open(encoding="utf-8-sig", newline="") as arquivo:
        return list(csv.DictReader(arquivo, delimiter=";"))


trechos = ler_csv(pasta_dados / "trechos_v3.csv")
indice_embeddings = ler_csv(
    pasta_dados / "indice_embeddings_v3.csv"
)
configuracao_base = ler_csv(
    pasta_dados / "configuracao_trechos_v3.csv"
)[0]
embeddings = np.load(
    pasta_dados / "embeddings_trechos_v3.npy",
    allow_pickle=False,
)

assert len(trechos) == len(indice_embeddings) == embeddings.shape[0]

for numero, (trecho, item) in enumerate(
    zip(trechos, indice_embeddings)
):
    assert int(item["linha"]) == numero
    assert item["id_trecho"] == trecho["id_trecho"]
    assert item["modelo"] == configuracao_base["modelo_embeddings"]
    assert item["sha256_texto"] == hashlib.sha256(
        trecho["texto_trecho"].encode("utf-8")
    ).hexdigest()

assert np.isfinite(embeddings).all()
assert np.allclose(
    np.linalg.norm(embeddings, axis=1), 1.0, atol=1e-5
)

print("Trechos conferidos:", len(trechos))
print("Embeddings:", embeddings.shape)
print("GPU disponível:", torch.cuda.is_available())

Trechos conferidos: 280
Embeddings: (280, 768)
GPU disponível: True


In [5]:
%pip install -q sentence-transformers transformers accelerate bitsandbytes gradio

In [6]:
from sentence_transformers import SentenceTransformer
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)

assert torch.cuda.is_available(), "Ative a GPU antes de continuar."

nome_modelo_busca = configuracao_base["modelo_embeddings"]
revisao_busca = configuracao_base["revisao_modelo"]

modelo_busca = SentenceTransformer(
    nome_modelo_busca,
    revision=(
        revisao_busca
        if revisao_busca != "Não identificada"
        else None
    ),
    device="cpu",
)

assert modelo_busca.get_embedding_dimension() == embeddings.shape[1]

nome_llm = "Qwen/Qwen2.5-7B-Instruct"

configuracao_4bits = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizador_llm = AutoTokenizer.from_pretrained(nome_llm)

modelo_llm = AutoModelForCausalLM.from_pretrained(
    nome_llm,
    quantization_config=configuracao_4bits,
    device_map={"": 0},
    dtype=torch.float16,
)

modelo_llm.eval()

print("Busca:", nome_modelo_busca)
print("Gerador:", nome_llm)
print("Quantização em 4 bits:", modelo_llm.is_loaded_in_4bit)
print("Dispositivo:", modelo_llm.get_input_embeddings().weight.device)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.68k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/908 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  436MB            

... (14 outputs left out)

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Busca: PORTULAN/serafim-100m-portuguese-pt-sentence-encoder-ir
Gerador: Qwen/Qwen2.5-7B-Instruct
Quantização em 4 bits: True
Dispositivo: cuda:0


In [7]:
def responder_rag(
    pergunta,
    trechos,
    embeddings,
    modelo_busca,
    modelo_llm,
    tokenizador_llm,
    k=3,
):
    import re
    import numpy as np
    import torch

    if not pergunta.strip():
        raise ValueError("Digite uma pergunta.")

    assert 1 <= k <= len(trechos)

    vetor = modelo_busca.encode(
        pergunta,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    pontuacoes = embeddings @ vetor
    indices = np.argsort(-pontuacoes)[:k]

    encontrados = [
        {
            **trechos[int(indice)],
            "similaridade": float(pontuacoes[indice]),
        }
        for indice in indices
    ]

    blocos = []

    for numero, trecho in enumerate(encontrados, start=1):
        metadados = [
            f"[{numero}] Arquivo: {trecho['arquivo']}",
            f"Trecho: {trecho['id_trecho']}",
        ]

        for campo, descricao in {
            "pagina": "Página",
            "mes": "Mês",
            "abrangencia_documentada": "Abrangência",
            "periodo_documentado": "Período",
            "limites_uso": "Limitações",
        }.items():
            if trecho.get(campo):
                metadados.append(f"{descricao}: {trecho[campo]}")

        blocos.append(
            "\n".join(metadados)
            + "\nConteúdo:\n"
            + trecho["texto_trecho"]
        )

    prompt = """
Responda à PERGUNTA em português brasileiro, de forma natural e breve,
usando exclusivamente as informações dos DOCUMENTOS.

Preserve os nomes dos menus e procedimentos.
Não invente etapas, documentos exigidos, botões ou explicações de siglas.
Respeite as limitações de abrangência e vigência das fontes.

Não escreva URLs nem marcadores de referência.
O sistema exibirá separadamente os documentos consultados.

Se os documentos não contiverem informação para responder, diga:
"Não encontrei informação suficiente nos documentos consultados."
""".strip()

    contexto = "\n\n".join(blocos)

    mensagens = [
        {"role": "system", "content": prompt},
        {
            "role": "user",
            "content": f"DOCUMENTOS:\n{contexto}\n\nPERGUNTA:\n{pergunta}",
        },
    ]

    entradas = tokenizador_llm.apply_chat_template(
        mensagens,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
    ).to(modelo_llm.get_input_embeddings().weight.device)

    tamanho_entrada = entradas["input_ids"].shape[1]

    assert (
        tamanho_entrada + 384
        <= modelo_llm.config.max_position_embeddings
    ), "A entrada excede o contexto disponível."

    with torch.inference_mode():
        saida = modelo_llm.generate(
            **entradas,
            max_new_tokens=384,
            do_sample=False,
            pad_token_id=tokenizador_llm.eos_token_id,
        )

    resposta = tokenizador_llm.decode(
        saida[0, tamanho_entrada:],
        skip_special_tokens=True,
    ).strip()

    conteudo = re.sub(r"\[[\d,\s]+\]", "", resposta)
    tem_conteudo = bool(re.search(r"[A-Za-zÀ-ÿ]", conteudo))

    return {
        "pergunta": pergunta,
        "resposta": (
            resposta if tem_conteudo
            else "Não consegui gerar uma resposta válida."
        ),
        "saida_original": resposta,
        "fontes_consultadas": encontrados,
        "modo": "gerada" if tem_conteudo else "revisar",
    }


teste_interface = responder_rag(
    "Como é calculada a média final das disciplinas?",
    trechos,
    embeddings,
    modelo_busca,
    modelo_llm,
    tokenizador_llm,
)

print(teste_interface["resposta"])
print("\nPrimeira fonte:", teste_interface["fontes_consultadas"][0]["id_trecho"])

A média final das disciplinas é calculada da seguinte forma:

MF = (0,4 x AP1) + (0,4 x AP2) + (0,2 x AC)

Onde:
- AP1 e AP2 são as notas obtidas nas Avaliações Parciais 1 e 2
- AC é a nota obtida na Avaliação Continuada

Primeira fonte: r090_t001


In [15]:
import gradio as gr
from html import escape
from urllib.parse import urlparse
from datetime import datetime, timezone

registros_interface = globals().get("registros_interface", [])


def formatar_fontes(fontes):
    blocos = []

    for numero, fonte in enumerate(fontes, start=1):
        titulo = fonte.get("titulo") or fonte["arquivo"]

        detalhes = [
            f"<h3>{numero}. {escape(titulo)}</h3>",
            f"<p>Arquivo: {escape(fonte['arquivo'])}</p>",
        ]

        if fonte.get("pagina"):
            detalhes.append(
                f"<p>Página: {escape(str(fonte['pagina']))}</p>"
            )

        if fonte.get("periodo_documentado"):
            detalhes.append(
                f"<p>Período: {escape(fonte['periodo_documentado'])}</p>"
            )

        url = (
            fonte.get("url_documento")
            or fonte.get("url_referencia")
            or fonte.get("url")
        )

        if url and urlparse(url).scheme in {"http", "https"}:
            detalhes.append(
                f'<p><a href="{escape(url, quote=True)}" '
                'target="_blank" rel="noopener noreferrer">'
                "Abrir fonte</a></p>"
            )

        detalhes.append(
            '<blockquote style="white-space: pre-wrap;">'
            + escape(fonte["texto_trecho"])
            + "</blockquote>"
        )

        blocos.append("\n".join(detalhes))

    return "\n<hr>\n".join(blocos)


def conversar(mensagem, historico):
    resultado = responder_rag_focado(
        mensagem,
        trechos,
        embeddings,
        modelo_busca,
        modelo_llm,
        tokenizador_llm,
    )

    registros_interface.append({
        "data_hora": datetime.now(timezone.utc).isoformat(),
        **resultado,
    })

    return (
        resultado["resposta"],
        formatar_fontes(resultado["fontes_consultadas"]),
    )


with gr.Blocks(title="FAQ acadêmico do Ibmec") as interface:
    gr.Markdown(
        "# FAQ acadêmico do Ibmec\n"
        "Protótipo acadêmico para consultar documentos do Ibmec "
        "Faria Lima. Faça uma pergunta completa a cada mensagem."
    )

    with gr.Accordion(
        "Fontes consultadas da última resposta",
        open=False,
    ):
        painel_fontes = gr.HTML()

    chat = gr.ChatInterface(
        fn=conversar,
        additional_outputs=[painel_fontes],
        examples=[
            "Como solicitar a carteirinha de estudante?",
            "Como é calculada a média final das disciplinas?",
            "Como enviar a documentação de estágio pelo SIA?",
        ],
        cache_examples=False,
        concurrency_limit=1,
    )

interface.queue(default_concurrency_limit=1).launch(
    share=True,
    inline=True,
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b7084ab356c10ba2b6.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [9]:
ultimo_teste = registros_interface[-1]

print("Pergunta:", ultimo_teste["pergunta"])
print("Trechos ativos:", len(trechos))
print("Fórmula presente na base:", any(
    trecho["id_trecho"] == "r090_t001"
    for trecho in trechos
))

for fonte in ultimo_teste["fontes_consultadas"]:
    print(
        "\n", fonte["id_trecho"],
        "|", fonte["arquivo"],
        "| similaridade:", round(fonte["similaridade"], 4),
    )
    print(fonte["texto_trecho"])

Pergunta: Como é calculada a nota final?
Trechos ativos: 280
Fórmula presente na base: True

 r090_t001 | GuiadoAlunoIbmecFariaLima.pdf | similaridade: 0.7441
Sistema de Avaliações
A sua média final é calculada a partir das notas obtidas nas AP1, AP2 e Avaliação Continuada, por meio do seguinte cálculo:
MF = (0,4 x AP1) + (0,4 x AP2) + (0,2 x AC)

 r022_t005 | GradManualAlunoIbmec.pdf | similaridade: 0.6315
(T+P+C): Teóricas, Práticas
e Campo, multiplicada pelo grau: (2.654 + 2.398 + 792) = 5.844;
2. Somar todos os resultados obtidos pela soma das horas (T+P+C): (320 + 340 + 120)
= 780;
3. Dividir o resultado do 1º passo pelo resultado do 2º passo: 5.844 / 780 = 7,5 resultado
final do CR Geral do aluno.
Importante: Apenas as disciplina Aprovadas (AP

 r022_t006 | GradManualAlunoIbmec.pdf | similaridade: 0.6299
5 resultado
final do CR Geral do aluno.
Importante: Apenas as disciplina Aprovadas (AP), Reprovadas por Notas (RN), Optativas
(O) e Eletivas (E) são calculadas. As disciplinas is

In [10]:
teste_contexto_reduzido = responder_rag(
    ultimo_teste["pergunta"],
    trechos,
    embeddings,
    modelo_busca,
    modelo_llm,
    tokenizador_llm,
    k=1,
)

comparacao_contexto = {
    "pergunta": ultimo_teste["pergunta"],
    "com_tres_trechos": ultimo_teste,
    "com_um_trecho": teste_contexto_reduzido,
}

print(teste_contexto_reduzido["resposta"])
print("\nFonte:", teste_contexto_reduzido["fontes_consultadas"][0]["id_trecho"])

A nota final é calculada da seguinte forma:
MF = (0,4 x AP1) + (0,4 x AP2) + (0,2 x AC)

Fonte: r090_t001


In [12]:
def responder_rag_focado(
    pergunta,
    trechos,
    embeddings,
    modelo_busca,
    modelo_llm,
    tokenizador_llm,
    k=3,
):
    import numpy as np

    if not pergunta.strip():
        raise ValueError("Digite uma pergunta.")

    assert 1 <= k <= len(trechos)

    vetor = modelo_busca.encode(
        pergunta,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    pontuacoes = embeddings @ vetor
    candidatos = np.argsort(-pontuacoes)[:k]

    registro_principal = trechos[
        int(candidatos[0])
    ]["id_registro"]

    selecionados = [
        int(indice)
        for indice in candidatos
        if trechos[int(indice)]["id_registro"] == registro_principal
    ]

    resultado = responder_rag(
        pergunta,
        [trechos[indice] for indice in selecionados],
        embeddings[selecionados],
        modelo_busca,
        modelo_llm,
        tokenizador_llm,
        k=len(selecionados),
    )

    resultado["politica_contexto"] = "mesmo_registro_do_primeiro_resultado"
    resultado["candidatos_recuperados"] = [
        {
            "id_trecho": trechos[int(indice)]["id_trecho"],
            "similaridade": float(pontuacoes[indice]),
        }
        for indice in candidatos
    ]

    return resultado

In [13]:
testes_contexto_focado = []

for pergunta_teste in [
    "Como é calculada a nota final?",
    "Como solicitar a carteirinha de estudante?",
    "Qual é a capital da França?",
]:
    resultado = responder_rag_focado(
        pergunta_teste,
        trechos,
        embeddings,
        modelo_busca,
        modelo_llm,
        tokenizador_llm,
    )

    testes_contexto_focado.append(resultado)

    print("\nPergunta:", pergunta_teste)
    print(resultado["resposta"])
    print("Trechos usados:", [
        fonte["id_trecho"]
        for fonte in resultado["fontes_consultadas"]
    ])


Pergunta: Como é calculada a nota final?
A nota final é calculada da seguinte forma:
MF = (0,4 x AP1) + (0,4 x AP2) + (0,2 x AC)
Trechos usados: ['r090_t001']

Pergunta: Como solicitar a carteirinha de estudante?
Para solicitar a carteirinha de estudante:

1. Acesse o portal SIA;
2. Vá até Atendimento > Requerimentos > Novo;
3. Escolha Certidão/Declaração/Documento;
4. Selecione Solicitar 1ª Via de Carteirinha de Acesso – Ibmec;
5. Preencha os dados corretamente e anexe sua foto no formato .JPG;
6. Pronto! Você será informado pelo SIA quando a carteirinha estiver pronta para ser retirada.
Trechos usados: ['r085_t007', 'r085_t008']

Pergunta: Qual é a capital da França?
Não encontrei informação suficiente nos documentos consultados.
Trechos usados: ['r043_t001']


In [14]:
interface.close()

Closing server running on port: 7860


In [16]:
ultimo = registros_interface[-1]

print("Pergunta:", ultimo["pergunta"])
print("Resposta:", ultimo["resposta"])
print("Política:", ultimo.get("politica_contexto", "Não registrada"))

mapa_trechos = {
    trecho["id_trecho"]: trecho
    for trecho in trechos
}

print("\nTrechos enviados ao modelo:", [
    fonte["id_trecho"]
    for fonte in ultimo["fontes_consultadas"]
])

print("\nCandidatos da busca:")

for candidato in ultimo.get("candidatos_recuperados", []):
    trecho = mapa_trechos[candidato["id_trecho"]]

    print(
        "\n", trecho["id_trecho"],
        "|", trecho["arquivo"],
        "| página:", trecho.get("pagina") or "Não se aplica",
    )
    print(trecho["texto_trecho"])

Pergunta: Como eu realizo o trancamento de matrícula?
Resposta: Para realizar o trancamento de matrícula, você deve seguir estes passos:

1. Acesse o Campus Virtual (SIA).
2. No menu "Atendimento > Requerimentos > Novo", selecione a categoria "Matrícula/Transferência > Entrevista Trancamento/Cancelamento de Matrícula".
3. Após a unidade entrar em contato, realize a entrevista e abra o requerimento de trancamento de matrícula.
4. Certifique-se de que sua matrícula está efetiva e vigente no momento da solicitação.
5. Verifique se você atende a todas as regras do contrato de prestação de serviço, além de respeitar as condições legais e prazos do calendário acadêmico.
Política: mesmo_registro_do_primeiro_resultado

Trechos enviados ao modelo: ['r028_t002', 'r028_t001']

Candidatos da busca:

 r028_t002 | GradManualAlunoIbmec.pdf | página: 38
/Transferência > Entrevista Trancamento/
Cancelamento de Matrícula”, após a unidade entrará em contato para realizar a
entrevista e abrir o requerimen

In [17]:
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from datetime import datetime, timezone
from importlib.metadata import version
import inspect
import textwrap
import hashlib
import json

raiz = Path("/content/projeto-pln")

# Exporta as funções que estão em uso nesta sessão.
codigo_rag = "\n\n".join(
    textwrap.dedent(inspect.getsource(funcao))
    for funcao in [responder_rag, responder_rag_focado]
)

codigo_fontes = (
    "from html import escape\n"
    "from urllib.parse import urlparse\n\n"
    + textwrap.dedent(inspect.getsource(formatar_fontes))
)

compile(codigo_rag, "rag.py", "exec")
compile(codigo_fontes, "fontes.py", "exec")

bibliotecas = [
    "numpy", "torch", "sentence-transformers",
    "transformers", "accelerate", "bitsandbytes", "gradio",
]
versoes = {nome: version(nome) for nome in bibliotecas}

nomes_registros = [
    "registros_interface",
    "testes_contexto_focado",
    "comparacao_contexto",
    "teste_interface",
]

registro_fechamento = {
    "data_exportacao": datetime.now(timezone.utc).isoformat(),
    "modelo_gerador": modelo_llm.config._name_or_path,
    "revisao_gerador": getattr(
        modelo_llm.config, "_commit_hash", None
    ),
    "quantizacao": "NF4, 4 bits, double quant; cálculo float16",
    "modelo_busca": modelo_busca.tokenizer.name_or_path,
    "base": "trechos_v3.csv",
    "quantidade_trechos": len(trechos),
    "politica_contexto": "Mesmo registro do primeiro resultado entre o top 3",
    "versoes": versoes,
    "registros_disponiveis": {
        nome: globals()[nome]
        for nome in nomes_registros
        if nome in globals()
    },
    "limites": [
        "Testes de desenvolvimento; não são avaliação independente.",
        "O histórico visual não é utilizado para interpretar novas perguntas.",
        "O contexto focado pode limitar perguntas que exigem fontes distintas.",
        "Fontes consultadas não equivalem a citações verificadas automaticamente.",
        "Caso de trancamento de uma disciplina ainda pendente de análise.",
    ],
}

gerados = {
    "src/rag.py": codigo_rag,
    "src/fontes.py": codigo_fontes,
    "requirements-colab.txt": "\n".join(
        f"{nome}==versao".replace("versao", versoes[nome])
        for nome in bibliotecas if nome != "torch"
    ) + "\n",
    "reports/etapa3/fechamento.json": json.dumps(
        registro_fechamento, ensure_ascii=False, indent=2
    ),
    "LEIA-ME_ETAPA3.md": (
        "# Entrega 3: protótipo RAG\n\n"
        "Busca com Serafim, geração com Qwen2.5-7B-Instruct "
        "em 4 bits e interface Gradio com fontes consultadas.\n\n"
        "## Execução\n\n"
        "1. Abra o notebook 06 no Colab e ative GPU T4.\n"
        "2. Use base_rag_v3.zip na célula de upload.\n"
        "3. Execute instalação, carga dos dados e dos modelos, "
        "funções e interface, nessa ordem.\n"
        "4. A sessão pode precisar baixar os modelos novamente.\n\n"
        "O PyTorch é fornecido pelo Colab; sua versão observada "
        "está no registro do experimento.\n\n"
        "Os notebooks 05 e 06 devem ser baixados separadamente "
        "pelo menu Arquivo do Colab.\n\n"
        "## Limitações\n\n"
        "Os testes disponíveis e as pendências estão em "
        "reports/etapa3/fechamento.json. Não afirmamos "
        "desempenho geral com base nesses exemplos.\n"
    ),
}

for nome, conteudo in gerados.items():
    caminho = raiz / nome
    caminho.parent.mkdir(parents=True, exist_ok=True)
    caminho.write_text(conteudo, encoding="utf-8")

arquivos_base = [
    "data/processed/trechos_v3.csv",
    "data/processed/embeddings_trechos_v3.npy",
    "data/processed/indice_embeddings_v3.csv",
    "data/processed/configuracao_trechos_v3.csv",
]

for nome in arquivos_base:
    assert (raiz / nome).is_file(), nome

# Mantém um pacote de entrada compatível com o notebook 06.
with ZipFile(
    raiz / "base_rag_v3.zip", "w", ZIP_DEFLATED
) as pacote:
    for nome in arquivos_base:
        pacote.write(raiz / nome, arcname=nome)

arquivos = arquivos_base + list(gerados) + ["base_rag_v3.zip"]

manifesto = {
    nome: hashlib.sha256((raiz / nome).read_bytes()).hexdigest()
    for nome in arquivos
}

(raiz / "manifesto_etapa3.json").write_text(
    json.dumps(manifesto, indent=2), encoding="utf-8"
)
arquivos.append("manifesto_etapa3.json")

destino = Path("/content/entrega_etapa3_rag.zip")

with ZipFile(destino, "w", ZIP_DEFLATED) as pacote:
    for nome in arquivos:
        pacote.write(raiz / nome, arcname=nome)

with ZipFile(destino) as pacote:
    assert pacote.testzip() is None
    for nome, assinatura in manifesto.items():
        assert hashlib.sha256(
            pacote.read(nome)
        ).hexdigest() == assinatura

print("Pacote criado e conferido:", destino.name)
print("Arquivos incluídos:", len(arquivos))

from google.colab import files
files.download(str(destino))

Pacote criado e conferido: entrega_etapa3_rag.zip
Arquivos incluídos: 11


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>